# Time-domain: An Absorbing Boundary

The PEC notebook showed the problem with a metal box: the radiated wave reflects
off the walls and returns, so anything after the first round trip is the box
rather than the antenna. This notebook replaces the wall with an absorbing layer,
so the wave can leave and the run can be extended indefinitely.

The mechanism has two parts, and only using both makes it work.

**Electric loss alone reflects.** Adding a conductivity $\sigma$ to the outer
cells does attenuate the wave — but it also changes the medium. A wave arriving at
the layer meets an impedance $\sqrt{\mu/\varepsilon}$ different from the vacuum it
came from, and part of it bounces back at the interface. The layer would absorb
what enters it while turning away much of what arrives.

**Matching fixes that.** Adding a *magnetic* loss $\sigma^*$ chosen so that

$$\frac{\sigma^*}{\mu} = \frac{\sigma}{\varepsilon}$$

leaves the impedance unchanged while both losses attenuate. A wave then enters the
layer without noticing the interface and decays inside it. $\sigma^*$ is not a
material property — there is no magnetic conductivity in nature. It is a numerical
device, the magnetic-current analogue of Ohmic loss, and it exists only inside the
layer.

The matching argument is a wave argument: it needs an impedance, hence a
propagating field. That distinguishes this case from the electrostatic notebooks,
where a similar-looking complex-stretched layer turned out to collapse into a
graded mesh, because with no wave there is no impedance to preserve and the
imaginary part had nothing to act on. Here it works as intended.

One limitation to note up front. This layer is matched at *normal* incidence.
Reflection grows as the angle of arrival becomes oblique, so waves striking the
edges and corners of the domain are absorbed less well than those hitting a face
head-on. A perfectly matched layer removes that restriction by stretching each
direction separately; it is considerably more work and is not used here.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..")) 
Pkg.instantiate() 
using FITToolbox;
using Statistics
using LinearAlgebra
using CairoMakie
using IterativeSolvers
using SparseArrays

function get_tangential_boundary(config::FITDomain)
    diag_vec = ones(Float64, 3*config.Np)
    # zero out the tangential entries; the rest pass through unchanged
    diag_vec[get_all_tangential_boundary_indices(config)] .= 0.0
    return spdiagm(0 => diag_vec)
end

## Domain and dipole

The domain is $220\;\text{mm}$, $20\;\text{mm}$ larger than in the PEC notebook,
because part of it is now spent on the absorbing layer: eight cells at each end of
every axis, so $16$ cells and $32\;\text{mm}$ per direction. Of the $110$ cells
along each axis, $94$ remain — a usable region of $188\;\text{mm}$, slightly
smaller than the $200\;\text{mm}$ before despite the larger grid. That is the
standing cost of an absorbing boundary: cells that hold no physics, only
attenuation.

The dipole sits at the centre, $94\;\text{mm}$ from the inner face of the layer.
That is a little over three wavelengths at $10\;\text{GHz}$, far enough that the
field reaching the layer is essentially propagating rather than reactive near
field. This matters for the layer's effectiveness: a matched conductive layer
absorbs propagating waves well but does little to evanescent content, so a source
placed close to the boundary would be absorbed much less cleanly than one placed
here.

Source placement follows the same two-step pattern as the earlier notebooks.
Current is a facet-integrated quantity, so the excitation must sit on a **dual
facet**, and those exist only at discrete positions. `get_index_entity` finds the
nearest one with a $z$-normal; `get_position_of_index` reads back where it actually
landed, which is what the plots mark.

In [ ]:
MyDomain = create_domain([220,220,220],[2,2,2];units="mm",σ=0,ε_r=1,μ_r=1); 

# These are the approximate coordinates where dipole should be placed
# However, the imprinted current lives on the dual facet and the correct
# position is found via function get_index_entity
x_dipole = MyDomain.nodes_u[end] / 2.0 * 1000 #convert to mm
y_dipole = MyDomain.nodes_v[end] / 2.0 * 1000 #convert to mm
z_dipole = MyDomain.nodes_w[end] / 2.0 * 1000 #convert to mm

index_dipole = get_index_entity(MyDomain, DualFacet(), Z(), x_dipole, y_dipole, z_dipole; units="mm", atol=0.005);

# For plotting, the dipole positions are updated
x_dipole, y_dipole, z_dipole = get_position_of_index(MyDomain, DualFacet(), index_dipole; units="mm");

## Building the layer

The conductivity is graded rather than constant. A step in $\sigma$ at the inner
face of the layer would itself be a material interface, and the wave would reflect
off it before ever reaching the lossy region — defeating the purpose. The quadratic
ramp

$$\sigma(\ell) = \sigma_\text{max}\left(\frac{\ell}{n}\right)^2$$

starts at $0.08\;\text{S/m}$ in the innermost cell and reaches $5\;\text{S/m}$ at
the wall, so the transition into the layer is gradual. The polynomial order is a
free parameter: too low and the inner face still reflects, too high and all the
attenuation is crowded into the last few cells where the grading is steepest.

Each face is written independently, and the corners are covered by two or three of
them. Taking the **maximum** rather than the sum keeps the profile monotone: a
corner cell gets the strongest of the contributions reaching it rather than three
times as much loss as a face cell at the same depth. Summing would create a
conductivity spike at the corners, which is exactly where the layer is already
weakest.

The count printed below should be $110^3 - 94^3 = 500{,}416$ cells — everything
outside the $94^3$ interior. A different number would mean a face was missed or
double-counted.

**The magnetic loss.** $\sigma^*$ is computed from the material matrices rather
than from the raw profile, so it inherits the same dual-facet averaging and
geometry as $\sigma$ itself. That keeps the matching condition $\sigma^*/\mu =
\sigma/\varepsilon$ true edge by edge, including at the layer's inner face where
the averaging smears $\sigma$ over one cell. Computing it from the profile
directly would satisfy the condition cell by cell but not edge by edge, and the
mismatch would show up as a small reflection.

The material matrices are built *after* the layer is written, since they are a
snapshot of `MyDomain.material` and would otherwise capture the vacuum state.

In [ ]:
# ===========================================================================
# Absorbing boundary layer
#
# The layer is matched: a graded electric conductivity σ is added in the outer
# cells, and a magnetic loss σ* chosen so that σ/ε = σ*/μ. That ratio is what
# makes the wave impedance √(μ/ε) unchanged inside the layer, so a wave entering
# it is attenuated rather than reflected. Unlike the electrostatic case, here
# there *is* a wave and an impedance, so the matching argument applies.
#
# The profile grows polynomially into the layer rather than jumping at its inner
# face, since a step in σ is itself a reflecting interface.
# ===========================================================================

n_abs     = 8          # layer thickness in cells
σ_max     = 5.0        # S/m at the outer face
abs_order = 2          # polynomial grading

# σ at each depth: 0 at the inner face of the layer, σ_max at the wall.
abs_profile = [σ_max * ((ℓ) / n_abs)^abs_order for ℓ in 1:n_abs]

Nu1, Nv1, Nw1 = MyDomain.Nu - 1, MyDomain.Nv - 1, MyDomain.Nw - 1
@assert 2n_abs < min(Nu1, Nv1, Nw1) "layer too thick for the domain"

md = MyDomain.material

# Write σ into all six faces. Corners are covered by more than one face, so the
# strongest contribution wins — taking the max rather than summing keeps the
# profile monotone into the corner instead of doubling it there.
for ℓ in 1:n_abs
    σℓ = abs_profile[n_abs - ℓ + 1]        # ℓ = 1 is the outermost cell

    for k in 1:Nw1, j in 1:Nv1
        md[ℓ, j, k, 1]           = max(md[ℓ, j, k, 1], σℓ)
        md[Nu1-ℓ+1, j, k, 1]     = max(md[Nu1-ℓ+1, j, k, 1], σℓ)
    end
    for k in 1:Nw1, i in 1:Nu1
        md[i, ℓ, k, 1]           = max(md[i, ℓ, k, 1], σℓ)
        md[i, Nv1-ℓ+1, k, 1]     = max(md[i, Nv1-ℓ+1, k, 1], σℓ)
    end
    for j in 1:Nv1, i in 1:Nu1
        md[i, j, ℓ, 1]           = max(md[i, j, ℓ, 1], σℓ)
        md[i, j, Nw1-ℓ+1, 1]     = max(md[i, j, Nw1-ℓ+1, 1], σℓ)
    end
end

println("absorbing layer: $n_abs cells, σ_max = $σ_max S/m, order $abs_order")
println("cells with loss: ", count(>(0), md[:, :, :, 1]))

# --- material matrices, built after the layer is in place --------------------
M_ε = get_permittivity(MyDomain)
M_ν = get_reluctivity(MyDomain)
M_σ = get_conductivity(MyDomain)

# Magnetic loss, matched to the electric one: σ*/μ = σ/ε, i.e. σ* = σ μ/ε.
# Evaluated per edge from the material matrices themselves so it follows the
# same averaging and geometry as everything else.
dσ  = diag(M_σ)
dε  = diag(M_ε)
dν  = diag(M_ν)
dμ  = 1.0 ./ dν
dμ[.!isfinite.(dμ)] .= 0.0                  # ghost facets carry no material

dσ_star = similar(dσ)
@inbounds for i in eachindex(dσ)
    dσ_star[i] = (dσ[i] > 0 && dε[i] > 0) ? dσ[i] * dμ[i] / dε[i] : 0.0
end

println("magnetic loss range: ", extrema(dσ_star))

## Operators

The curl operators carry no material information at all. $\mathbf{C}$ maps primal
edges to primal facets, $\tilde{\mathbf{C}} = \mathbf{C}^\mathsf{T}$ maps dual
edges to dual facets, and both contain nothing but $\pm 1$. They are unaffected by
the absorbing layer: attenuation is a material property, so the entire boundary
treatment lives in $\mathbf{M}_\sigma$ and the coefficients derived from it, while
the topology is identical to the free-space case.

That separation is what makes the layer easy to add. Nothing about the operators,
the time step, or the structure of the update changes — only the numbers in the
diagonal matrices.

The material matrices repeat what the previous cell already built. They are
rebuilt here for clarity rather than necessity; the values are the same, since
`MyDomain.material` has not changed since.

In [ ]:
M_ν = get_reluctivity(MyDomain);
M_σ = get_conductivity(MyDomain)
M_ε = get_permittivity(MyDomain)

C = get_curl(MyDomain,Primal());
C̃ = get_curl(MyDomain, Dual());

## Excitation and time step

A sinusoidal current at $10\;\text{GHz}$ driven through the single dual facet
located earlier. The wavelength is $\lambda = 30\;\text{mm}$, resolved by fifteen
cells, and the time step is $99\%$ of the Courant limit — about $3.81\;\text{ps}$,
giving $26$ steps per period.

The layer does not change the time step. That is worth checking rather than
assuming, because a conductive region normally can: the relevant ratio is
$\sigma\Delta t/\varepsilon_0$, and at $\sigma_\text{max} = 5\;\text{S/m}$ it comes
to about $2.2$. The dielectric relaxation time in the outermost cells is
$\varepsilon_0/\sigma \approx 1.8\;\text{ps}$, comparable to the step itself — so
the loss is significant per step but not stiff. A conventional semi-implicit update
would cope here; the exponential form used below is not strictly necessary at this
conductivity, but it costs nothing and removes the question entirely.

Contrast the scattering notebook, where the sphere's copper gave
$\sigma\Delta t/\varepsilon \approx 2.5\times10^{7}$ and the standard update failed
outright. The absorbing layer is deliberately mild: its job is to attenuate a wave
over eight cells, not to act as a conductor, and $\sigma_\text{max}$ is chosen for
that. Making it far more conductive would not absorb better — it would reflect,
since a large impedance mismatch at the layer's inner face is exactly what the
matching is designed to avoid.

As before, a sine switched on at $t = 0$ has a discontinuous derivative and
therefore broadband content the grid cannot resolve. It appears as a faint
precursor ahead of the wavefront and sits well below the field of interest.

In [ ]:
#FDTD excitation
f = 10e9           # Operating frequency in Hz
ω = 2π * f         # Angular frequency
A_p = 1            # Peak amplitude

I_src(t) = A_p * sin(ω * t)

Δt = get_vacuum_cfl_time(MyDomain) * 0.99;

## The time loop

The update pair is the same leapfrog as before, with both loss terms now
integrated exponentially. Faraday's law gains the magnetic sink,

$$\frac{d\hat{\hat{b}}}{dt} = -\mathbf{C}\hat{e} - \mathbf{M}_{\sigma^*}\hat{h}
= -\mathbf{C}\hat{e} - \mathbf{M}_{\sigma^*}\mathbf{M}_{\nu}\hat{\hat{b}}$$

and Ampère's law keeps the electric one,

$$\mathbf{M}_\varepsilon \frac{d \hat{e}}{d t} + \mathbf{M}_\sigma \hat{e}
  = \tilde{\mathbf{C}}\hat{h} - \hat{\hat{\jmath}} .$$

Both have the same form — a decay term plus a source — so both are solved exactly
over one step:

$$\hat{\hat{b}} \leftarrow D_\beta\, \hat{\hat{b}} - \beta_b\, \mathbf{C}\hat{e},
\qquad
\hat{e} \leftarrow D_\alpha\, \hat{e} + \beta_e\left(\tilde{\mathbf{C}}\hat{h} - \hat{\hat{\jmath}}\right).$$

The coefficients are computed once, before the loop. Outside the layer $\sigma = 0$,
giving $D_\alpha = D_\beta = 1$, $\beta_e = \Delta t/\varepsilon$ and
$\beta_b = \Delta t$ — the lossless update recovered exactly. The two regimes
therefore share one line of code and join continuously at the layer's inner face,
with no branch inside the loop and no special casing at the interface.

**Why the magnetic side is written in terms of $\hat{\hat{b}}$.** The decay rate for
the flux is $a = \sigma^*\nu$, which requires the flux and the reluctivity
separately. Folding $\hat{h} = \mathbf{M}_\nu \hat{\hat{b}}$ into a single variable, as
the lossless notebook did, would hide the quantity the exponential acts on. The
extra array costs one pass per step and makes the structure explicit.

**A PEC wall still sits behind the layer.** The absorbing region attenuates but does
not eliminate — some field reaches the outermost cells, and something has to be
imposed there. By that point it is small enough that the reflection is negligible,
which is precisely the arrangement the layer exists to create.

The run covers $500\;\text{ps}$, or $131$ steps. The wave reaches the inner face of
the layer at about $314\;\text{ps}$ and the outer wall at $367\;\text{ps}$, so the
final frames show whether the layer absorbs. Had the boundary been PEC throughout,
the reflection would return to the source at $734\;\text{ps}$ — beyond this window,
so the comparison with the PEC notebook is best made on the frames near the
boundary rather than at the centre.

In [ ]:
# ===========================================================================
# FDTD leapfrog with a matched absorbing layer
#
# Everything heavy is inside a function. At notebook top level each reference to
# a global is a dynamic dispatch, so a loop over six million elements that
# should take milliseconds takes minutes instead — which is what happens if the
# setup below is written as a bare `for` in a cell.
# ===========================================================================

"""
    fdtd_coefficients(M_ε, M_ν, M_σ, Δt)

Per-edge update coefficients for the exponentially fitted leapfrog.

Electric side, integrating ε ė + σ e = s exactly over one step:
    e ← D_α e + β_e s,   D_α = exp(-σΔt/ε),   β_e = (1 - D_α)/σ
As σ → 0 this tends to e + (Δt/ε) s, so the vacuum case is the same formula
with D_α = 1 and β_e = Δt/ε — no branch is needed in the loop.

Magnetic side: a conductive layer alone reflects, because the wave meets a
medium of different impedance √(μ/ε). Adding a magnetic loss σ* with
    σ*/μ = σ/ε
leaves the impedance unchanged, so the layer attenuates without reflecting.
σ* is not a material property — there is no magnetic conductivity in nature.
It is a numerical device, the magnetic-current analogue of Ohmic loss, and it
exists only inside the absorbing layer. It enters Faraday's law as a sink,
    ḃ = -C ê - σ* ĥ = -C ê - (σ* ν) b̂̂
integrated the same way with a = σ*ν and β_b = (1 - exp(-aΔt))/a → Δt.
"""
function fdtd_coefficients(M_ε, M_ν, M_σ, Δt)
    dε = diag(M_ε)
    dν = diag(M_ν)
    dσ = diag(M_σ)
    n  = length(dε)

    D_α = ones(n);  β_e = zeros(n)
    D_β = ones(n);  β_b = fill(Δt, n)
    dσ_star = zeros(n)

    @inbounds for i in 1:n
        ε, ν, σ = dε[i], dν[i], dσ[i]

        # Electric. Dead edges have ε = 0 and no equation, so leave β_e at 0
        # rather than dividing by zero.
        if ε > 0
            if σ > 0
                D_α[i] = exp(-Δt * σ / ε)
                β_e[i] = (1 - D_α[i]) / σ
            else
                β_e[i] = Δt / ε
            end
        end

        # Magnetic. σ* = σ·μ/ε = σ/(ν·ε), so μ never has to be formed and the
        # Inf at ghost facets (ν = 0) never arises.
        if σ > 0 && ε > 0 && ν > 0
            dσ_star[i] = σ / (ν * ε)
            a = dσ_star[i] * ν
            D_β[i] = exp(-Δt * a)
            β_b[i] = (1 - D_β[i]) / a
        end
    end

    return (; dν, dσ, dσ_star, D_α, β_e, D_β, β_b)
end

"""
    run_fdtd(C, C̃, coef, pec, index_dipole, I_src, Δt, nsteps, targets)

Leapfrog time stepping. ê lives at half steps, b̂̂ at whole steps. Returns the
snapshots listed in `targets` — storing every step would cost
nsteps × n_e × 8 bytes per field, which is far too much on a large grid.
"""
function run_fdtd(C, C̃, coef, pec, index_dipole, I_src, Δt, nsteps, targets)
    (; dν, D_α, β_e, D_β, β_b) = coef

    n_e, n_b = size(C, 2), size(C, 1)
    ê   = zeros(n_e)          # edge voltages
    b̂̂   = zeros(n_b)          # magnetic flux through primal facets
    ĥ   = zeros(n_b)          # magnetic voltages along dual edges
    src = zeros(n_e)          # C̃ĥ - ĵ̂, right-hand side of the e-update
    tmp = zeros(n_b)

    ê_history, ê_times = Vector{Vector{Float64}}(), Float64[]
    b̂̂_history, b̂̂_times = Vector{Vector{Float64}}(), Float64[]

    t = Δt / 2
    for step in 1:nsteps
        if step in targets
            push!(ê_history, copy(ê)); push!(ê_times, t)
            push!(b̂̂_history, copy(b̂̂)); push!(b̂̂_times, t - Δt/2)
        end

        # Faraday with magnetic loss: b̂̂ ← D_β b̂̂ - β_b (C ê)
        mul!(tmp, C, ê)
        @. b̂̂ = D_β * b̂̂ - β_b * tmp

        # Constitutive: ĥ = M_ν b̂̂
        @. ĥ = dν * b̂̂

        # Ampère with the source: src = C̃ ĥ - ĵ̂
        mul!(src, C̃, ĥ)
        src[index_dipole] -= I_src(t)

        # Electric loss and the PEC wall, fused into one pass.
        @. ê = pec * (D_α * ê + β_e * src)

        t += Δt
    end

    return ê_history, ê_times, b̂̂_history, b̂̂_times
end


# --- run ---------------------------------------------------------------------
@assert Δt > 0

coef = fdtd_coefficients(M_ε, M_ν, M_σ, Δt)
pec  = diag(get_tangential_boundary(MyDomain))

nsteps  = ceil(Int, (5e-10 - Δt/2) / Δt)
targets = Set(round.(Int, LinRange(max(1, nsteps ÷ 10), nsteps, 9)))

println("electric loss range: ", extrema(coef.dσ))
println("magnetic loss range: ", extrema(coef.dσ_star))
println("CFL limit ", get_vacuum_cfl_time(MyDomain), " s, using Δt = ", Δt,
        " over ", nsteps, " steps")

@time ê_history, ê_times, b̂̂_history, b̂̂_times =
    run_fdtd(C, C̃, coef, pec, index_dipole, I_src, Δt, nsteps, targets);

## Snapshots

Nine frames on a plane through the dipole. The sampling grid spans the whole
domain, so the outermost $16\;\text{mm}$ on each side is the absorbing layer — the
field there is being attenuated deliberately, and what happens inside it is not
physics but the boundary doing its job.

**The signed $E_z$ component**, shown first, is where the boundary's behaviour is
clearest. Successive wavefronts appear as alternating red and blue rings expanding
outward. The test is what happens when they reach the layer at about
$314\;\text{ps}$: with a working absorber they simply fade as they enter it, and
the rings behind continue outward undisturbed. A reflection would show as a second
set of rings travelling inward and interfering with the first, breaking the radial
symmetry — which is exactly what the PEC notebook shows at the same time.

**The magnitude**, logarithmic and clipped to four decades below the peak, shows
the attenuation profile directly: the field should fall steadily through the layer
rather than dropping abruptly at its inner face. An abrupt drop would mean the
grading is too steep and the layer's inner face is itself reflecting.

The clipping matters here for the usual reason. Below about $10^{-6}$ of the peak
the plot shows the numerical precursor rather than the field — each step couples
only nearest neighbours, so the reachable region after $n$ steps is a diamond
expanding faster than the physical wave, and what lies inside it but outside the
wavefront is switch-on transient and dispersion.

Comparing against the PEC notebook is the point of the exercise. There, the field
near the walls grows as reflections accumulate; here it should stay at the level
set by the outgoing wave alone. The clearest evidence is in the last two or three
frames, after the wave has reached the boundary and before a reflection would have
had time to return.

In [ ]:
# Sampling grid in mm, one node in from each boundary so the interpolation always
# has a full stencil. MyDomain stores metres, hence the scaling.
mm = 1e-3
xs = LinRange(MyDomain.nodes_u[2]/mm, MyDomain.nodes_u[end-1]/mm, 200)
ys = LinRange(MyDomain.nodes_v[2]/mm, MyDomain.nodes_v[end-1]/mm, 200)

# Interpolate ê onto the cut plane at z_dipole and reduce each triple of field
# components with f — b -> hypot(b...) for the magnitude, last for E_z.
sample_slice(ê, f) =
    [f(FITToolbox.interpolate(MyDomain, PrimalEdge(), ê, x, y, z_dipole; units="mm"))
     for x in xs, y in ys]

# Nine snapshots in a 3×3 grid sharing one colour range, so a brighter panel means
# a stronger field rather than a rescaled axis. Axis labels only on the outer row
# and column, to keep the grid readable.
function snapshot_grid(slices, times, colorrange, colormap, label)
    fig = Figure(size = (900, 950))
    Label(fig[0, 1:3],
          "Dipole radiation in the x–y plane at z = $(round(z_dipole, digits=1)) mm";
          fontsize = 18, padding = (0, 0, 8, 0))

    for (n, slice) in enumerate(slices)
        row, col = fldmod1(n, 3)
        ax = Axis(fig[row, col];
                  title  = "t = $(round(times[n] * 1e12, digits=2)) ps",
                  aspect = DataAspect(),
                  xlabel = row == 3 ? "x (mm)" : "",
                  ylabel = col == 1 ? "y (mm)" : "",
                  xticklabelsvisible = row == 3,
                  yticklabelsvisible = col == 1)

        heatmap!(ax, xs, ys, slice; colormap, colorrange)
        scatter!(ax, [x_dipole], [y_dipole];
                 color = :red, markersize = 8, strokewidth = 1, strokecolor = :white)
    end

    Colorbar(fig[1:3, 4]; colorrange, colormap, label)
    return fig
end


# --- magnitude, logarithmic ---------------------------------------------------
# The floor prevents log10(0) where the field passes through zero.
mag = [log10.(max.(sample_slice(ê, b -> hypot(b...)), 1e-15)) for ê in ê_history]
vmax = maximum(maximum, mag)
fig_mag = snapshot_grid(mag, ê_times, (vmax - 4, vmax), :viridis, "log₁₀(|E|) (V/m)")

# --- z-component, signed ------------------------------------------------------
# The dipole points along z, so E_z carries the sign alternation of the wave. A
# diverging colormap on a symmetric range makes the wavefronts far more legible
# than a magnitude, which is positive everywhere. The 99th percentile rather than
# the maximum keeps the near-field spike from flattening everything else.
ez   = [sample_slice(ê, last) for ê in ê_history]
vabs = quantile(abs.(vcat(vec.(ez)...)), 0.99)
fig_ez = snapshot_grid(ez, ê_times, (-vabs, vabs), :RdBu, "E_z (V/m)")

fig_ez

In [ ]:
fig_mag